# Neural Machine Translation

This week and the next, we will build a neural machine translation model based on the sequence-to-sequence (seq2seq) models proposed by Sutskever et al., 2014 and Cho et al., 2014. The seq2seq model is widely used in Machine Translation systems such as Google’s neural machine translation system (GNMT) (Wu et al., 2016).

In today’s lab and the one next week, we will explore the seq2seq model, as well as attention in machine translation.

For training and evaluating our mode, we will use the English-Vietnamese parallel corpus of TED talks provided by the IWSLT Evaluation Campaign. For our tasks, we will translate from Vietnamese into English.

The parallel corpus has been provided for you:
1. **data.30.vi** - a file where each line contains a Vietnamese sentence to be translated (i.e. the source sentences)
2. **data.30.en** - a file where each line contains an English sentence corresponding to the Vietnamese sentence in the same line position. (i.e. the target sentences)


In [1]:
!wget 'https://github.com/juntaoy/ECS7001_LAB_DATASETS/raw/refs/heads/main/NMT_data.zip'
!unzip -o NMT_data.zip

--2026-03-15 06:44:35--  https://github.com/juntaoy/ECS7001_LAB_DATASETS/raw/refs/heads/main/NMT_data.zip
Resolving github.com (github.com)... 20.26.156.215
Connecting to github.com (github.com)|20.26.156.215|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/juntaoy/ECS7001_LAB_DATASETS/refs/heads/main/NMT_data.zip [following]
--2026-03-15 06:44:35--  https://raw.githubusercontent.com/juntaoy/ECS7001_LAB_DATASETS/refs/heads/main/NMT_data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4815130 (4.6M) [application/zip]
Saving to: ‘NMT_data.zip.4’

NMT_data.zip.4      100%[===================>]   4.59M  --.-KB/s    in 0.01s   

2026-03-15 06:44:35 (445 MB/s) - ‘NMT_data.zip.4’ saved [4815130/481

In [2]:
'Lets first install the `Sacrebleu` (https://github.com/mjpost/sacrebleu) package for BLEU computation.'

'Lets first install the `Sacrebleu` (https://github.com/mjpost/sacrebleu) package for BLEU computation.'

In [3]:
!pip install sacrebleu

Defaulting to user installation because normal site-packages is not writeable


## Overview
This script defines a total of three classes: the main class (`NmtModel`), the attention layer class (`AttentionLayer`) and a helper class (`LanguageDict`). The `NmtModel` class contains most of the code of the NMT system and is the one you are asked to complete for Task 1 and 2. The `AttentionLayer` class is a custom layer to implement the attention mechanism, Task 3 is to finish this class. `LanguageDict` is a class that stores resources related to languages, such as vocab, word2ids, etc. The code for this class is provided.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import collections
import numpy as np
import time
from sacrebleu import corpus_bleu

SOURCE_PATH = 'data.30.vi'
TARGET_PATH = 'data.30.en'

[HAMI-core Msg(958:140078627843392:libvgpu.c:839)]: Initializing.....


## The `LanguageDict` class stores the language resources
This class has only an initialisation method. The method takes a corpus as the input and builds the vocab and word2ids for the language.

In [5]:
class LanguageDict():
  def __init__(self, sents):
    word_counter = collections.Counter(tok.lower() for sent in sents for tok in sent)

    self.vocab = []
    self.vocab.append('<pad>') #zero paddings
    self.vocab.append('<unk>')
    # add only words that appear at least 10 times in the corpus
    self.vocab.extend([t for t,c in word_counter.items() if c > 10])

    self.word2ids = {w:id for id, w in enumerate(self.vocab)}
    self.UNK = self.word2ids['<unk>']
    self.PAD = self.word2ids['<pad>']

## The `load_dataset()` method creates train/dev/test batches
The method reads the given file and loads the first max_num_examples sentences and split them into train/dev/test dataset:

In [6]:
def pad_sequences(seq_list, max_len=None, pad_value=0):
    """
    A simple PyTorch-like pad_sequences function.
    seq_list: List of lists of token IDs.
    max_len : If None, will use the length of the longest sequence.
    pad_value: ID to use for padding.
    Returns a 2D NumPy array with shape [batch_size, max_length].
    """
    if max_len is None:
        max_len = max(len(seq) for seq in seq_list)
    padded = []
    for seq in seq_list:
        seq = seq[:max_len]
        padded.append(seq + [pad_value]*(max_len - len(seq)))
    return np.array(padded)

In [7]:
def load_dataset(source_path,target_path, max_num_examples=30000):
  ''' This helper method reads from the source and target files to load max_num_examples
  sentences split them into train, development and testing and return relevant data.
  Inputs:
    source_path (string): the full path to the source data, SOURCE_PATHf
    target_path (string): the full path to the target data, TARGET_PATH
  Returns:
    train_data (list): a list of 3 elements: source_words, target words, target word labels
    dev_data (list): a list of 2 elements - source words, target word labels
    test_data (list): a list of 2 elements - source words, target word labels
    source_dict (LanguageDict): a LanguageDict object for the source language, Vietnamese.
    target_dict (LanguageDict): a LanguageDict object for the target language, English.
  '''
  # source_lines/target lines are list of strings
  # such that each string is a sentence in the corresponding file
  source_lines = open(source_path).readlines()
  target_lines = open(target_path).readlines()
  assert len(source_lines) == len(target_lines)
  if max_num_examples > 0:
    max_num_examples = min(len(source_lines), max_num_examples)
    source_lines = source_lines[:max_num_examples]
    target_lines = target_lines[:max_num_examples]

  # strip trailing/leading whitespaces and tokenize each sentence
  source_sents = [[tok.lower() for tok in sent.strip().split(' ')] for sent in source_lines]
  target_sents = [[tok.lower() for tok in sent.strip().split(' ')] for sent in target_lines]
  # for the target sentences, add <start> and <end> tokens to each sentence
  for sent in target_sents:
    sent.append('<end>')
    sent.insert(0,'<start>')

  # create the LanguageDict objects for each file
  source_lang_dict = LanguageDict(source_sents)
  target_lang_dict = LanguageDict(target_sents)


  # for the source sentences:
  # we'll use this proportion to split into train/dev/test
  unit = len(source_sents)//10
  # get the sents-as-ids for each sentence
  source_words = [[source_lang_dict.word2ids.get(tok,source_lang_dict.UNK) for tok in sent] for sent in source_sents]
  # 8 parts (80%) of the sentences go to the training data and are padded up to the maximum sentence length
  source_words_train = pad_sequences(source_words[:8*unit])
  # 1 part (10%) of the sentences go to the dev data and are padded up to the up to the maximum sentence length
  source_words_dev = pad_sequences(source_words[8*unit:9*unit])
  # 1 part (10%) of the sentences go to the test dataand are padded up to the up to the maximum sentence length
  source_words_test = pad_sequences(source_words[9*unit:])


  eos = target_lang_dict.word2ids['<end>']
  # for each sentence, get the word index for the tokens from <start> to up to but not including <end>,
  target_words = [[target_lang_dict.word2ids.get(tok,target_lang_dict.UNK) for tok in sent[:-1]] for sent in target_sents]
  # select the training set and pad the sentences
  target_words_train = pad_sequences(target_words[:8*unit])
  # the label for each target word is the next word, we also add <end> as the last token
  target_words_train_labels = [sent[1:]+[eos] for sent in target_words[:8*unit]]
  # pad the labels. Dim = [num_sents, max_sent_length]
  target_words_train_labels = pad_sequences(target_words_train_labels)
  # expand one dimension at the end for the loss computation. Dim = [num_sents, max_sent_length, 1].
  target_words_train_labels = np.expand_dims(target_words_train_labels,axis=2)

  # get the labels for the dev and test data. No need for inputs here and no need to expand dimensions
  target_words_dev_labels = pad_sequences([sent[1:] + [eos] for sent in target_words[8 * unit:9 * unit]])
  target_words_test_labels = pad_sequences([sent[1:] + [eos] for sent in target_words[9 * unit:]])

  # our final data
  train_data = [source_words_train,target_words_train,target_words_train_labels]
  dev_data = [source_words_dev,target_words_dev_labels]
  test_data = [source_words_test,target_words_test_labels]

  return train_data,dev_data,test_data,source_lang_dict,target_lang_dict

## The `AttentionLayer` class creates a custom layer for attention

The class takes two inputs: the `encoder_outputs` and the `decoder_outputs` and returns a `new_decoder_outputs` that leverages the `decoder_outputs` with the `encoder_outputs`.

This class contains three methods. The first one is used for passing the mask to the next layer. The mask is originally created by the `Embedding` layer with the `mask_zero` attribute set to `True`, so that the padding is not taken into account in the computations of loss or by LSTM layers. So, in this first method we return the mask for the `decoder_outputs`. The second method computes the output shape of our layer. The output shape of the layer is the same to the `decoder_outputs` in the first two dimensions and for the last dimension the embedding dimension is doubled.

The third method is the main method for the layer, and also the one you will need to implement for your Task 3. We will come back to this later.


In [8]:
class AttentionLayer(nn.Module):
    """
    Custom layer implementing Luong attention.
    """
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def forward(self, encoder_outputs, decoder_outputs, mask_src=None, mask_tgt=None):
        """
        encoder_outputs : [batch_size, max_source_length, hidden_size]
        decoder_outputs : [batch_size, max_target_length, hidden_size]

            1) Transposing decoder outputs to shape [batch_size, hidden_size, max_target_length]
            2) Doing a batch matrix multiplication with encoder_outputs ->
               shape [batch_size, max_source_length, max_target_length]
            3) Softmax over max_source_length dimension
            4) Weighted sum over encoder_outputs to produce context vectors.
            5) Concatenate them with the original decoder outputs ->
               final shape [batch_size, max_target_length, hidden_size*2]
        """


        """
        Task 3: Adding attention

        Begin
        """
           # ---------------------------------------------------------
        # TEMPLATE-STYLE LUONG ATTENTION (expand-dims + multiply + sum)
        # ---------------------------------------------------------
        # encoder_outputs: [B, S, H]
        # decoder_outputs: [B, T, H]
        #
        # We will compute:
        #   scores[b, s, t] = dot(encoder_outputs[b, s, :], decoder_outputs[b, t, :])
        #   attn[b, s, t] = softmax(scores over s)   (so weights sum to 1 over source positions)
        #   context[b, t, :] = sum_s attn[b, s, t] * encoder_outputs[b, s, :]
        # ---------------------------------------------------------

        # 1) Transpose decoder outputs to [B, H, T]
        decoder_outputs_t = decoder_outputs.permute(0, 2, 1)  # [B, H, T]

        # 2) Dot product scores => [B, S, T]
        luong_score = torch.bmm(encoder_outputs, decoder_outputs_t)  # [B, S, T]

        # 3) Softmax over source length dimension (S)
        attn = F.softmax(luong_score, dim=1)  # [B, S, T]

        # 4) Weighted sum of encoder_outputs to get context vectors
        # Expand dims so broadcasting works:
        #   attn_expanded:    [B, S, T, 1]
        #   enc_expanded:     [B, S, 1, H]
        attn_expanded = attn.unsqueeze(-1)            # [B, S, T, 1]
        enc_expanded  = encoder_outputs.unsqueeze(2)  # [B, S, 1, H]

        # Elementwise multiply -> [B, S, T, H]
        weighted_enc = attn_expanded * enc_expanded

        # Sum over source positions S -> context: [B, T, H]
        encoder_vector = weighted_enc.sum(dim=1)      # [B, T, H]




        """
        End Task 3
        """

        # 5) Concatenate
        new_decoder_outputs = torch.cat([decoder_outputs, encoder_vector], dim=-1)
        return new_decoder_outputs

## NmtModel class `__init__()` method: initialises the network parameters.

This method takes three arguments. The first two are instances of `LanguageDict`, one for the source language (Vietnamese) and one for the target language (English); the third argument is a boolean variable (`use_attention`) that indicates which model (attention/basic) should be used.

It then creates all the layers will be used in later stages.



In [9]:
class NmtModel(nn.Module):
    def __init__(self, source_dict, target_dict, use_attention):
        """
        Initializes the NMT Model hyperparameters and layers.
        """
        super().__init__()
        self.source_dict = source_dict
        self.target_dict = target_dict
        self.use_attention = use_attention

        # Hyperparams
        self.hidden_size = 200
        self.embedding_size = 100
        self.hidden_dropout_rate = 0.2
        self.embedding_dropout_rate = 0.2
        self.batch_size = 100
        self.max_target_step = 30

        # Special tokens
        self.SOS = target_dict.word2ids['<start>']
        self.EOS = target_dict.word2ids['<end>']

        # Vocab sizes
        self.vocab_source_size = len(source_dict.vocab)
        self.vocab_target_size = len(target_dict.vocab)

        print(f"number of tokens in source: {self.vocab_source_size}, "
              f"number of tokens in target: {self.vocab_target_size}")


        """
        Task 1: Implementing the encoder 1/2

        Begin
        """
        # -----------------------------
        # Task 1 (1/2): Embeddings
        # -----------------------------
        # Vietnamese (source) embedding
        self.embedding_source = nn.Embedding(
            num_embeddings=self.vocab_source_size,
            embedding_dim=self.embedding_size,
            padding_idx=self.source_dict.PAD
        )

        # English (target) embedding
        self.embedding_target = nn.Embedding(
            num_embeddings=self.vocab_target_size,
            embedding_dim=self.embedding_size,
            padding_idx=self.target_dict.PAD
        )

        # -----------------------------
        # Task 1 (1/2): Encoder LSTM
        # -----------------------------
        # Takes source word vectors and produces:
        #  - encoder_outputs: [B, src_len, hidden_size]
        #  - (enc_h, enc_c):  [1, B, hidden_size] each
        self.encoder_lstm = nn.LSTM(
            input_size=self.embedding_size,
            hidden_size=self.hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False
            # NOTE: no dropout here because num_layers=1
        )



        """
        End Task 1 1/2
        """

        # Decoder LSTM
        self.decoder_lstm = nn.LSTM(
            input_size=self.embedding_size,
            hidden_size=self.hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=self.hidden_dropout_rate,
            bidirectional=False
        )

        # Attention (if use_attention)
        if self.use_attention:
            self.decoder_attention = AttentionLayer()

        # Final projection
        # If attention, hidden_size * 2, else hidden_size
        if self.use_attention:
            self.decoder_dense = nn.Linear(self.hidden_size*2, self.vocab_target_size)
        else:
            self.decoder_dense = nn.Linear(self.hidden_size, self.vocab_target_size)

## NmtModel `forward()`, `decode_step()` and `encode()` methods:  builds the PyTorch models for training and inference.
The method first creates the inputs for both training and inference models, which include the source/target sentence batches; The inputs specifically used for the inference models are defined later.

Task 1 will be to create embeddings for both source/target languages as well as the encoder. We will discuss this in a later section.

After that, we define the decoder used for the training. In NMT separate decoders are often used for training and inference. During training, we feed the ground truth tokens into the decoder (teacher forcing), hence we process all tokens in the sentences in a single step. During inference, the system processes one token at a time, and the token predicted at the current step will be used as the input for the next step.  More specifically, the size of `target_words` will be `[batch, max_sent_len]` during training and `[batch, 1]` during inference. The training and inference models behave slightly differently, but they share all the layers (`decoder_lstm, decoder_attention and decoder_dense`);

Task 2 will be to implement the decoder for inference. We will discuss this later.

In [10]:
class NmtModel(NmtModel):
    def forward(self, source_words, target_words):
        """
        Forward pass for training:
          1) Encode the source sentences using the encoder LSTM.
          2) Use the final encoder state to initialize the decoder's hidden state.
          3) Feed all target words into the decoder LSTM in one go (teacher forcing).
          4) (Optional) apply the attention layer between the decoder outputs and the encoder outputs.
          5) Project the decoder outputs to vocabulary logits with self.proj.
        """

        """
        Task 1: Implementing the encoder 2/2

        Begin
        """
        # -----------------------------
        # Task 1 (2/2): Embedding lookup
        # -----------------------------
        source_words_embeddings = F.dropout(
            self.embedding_source(source_words),
            p=self.embedding_dropout_rate,
            training=self.training
        )

        target_words_embeddings = F.dropout(
            self.embedding_target(target_words),
            p=self.embedding_dropout_rate,
            training=self.training
        )

        # -----------------------------
        # Task 1 (2/2): Encode source sentence
        # -----------------------------
        encoder_outputs, (enc_h, enc_c) = self.encoder_lstm(source_words_embeddings)

        # -----------------------------
        # Decoder for training (teacher forcing)
        # -----------------------------
        decoder_outputs, (dec_h, dec_c) = self.decoder_lstm(
            target_words_embeddings,
            (enc_h, enc_c)
        )

        # Optional attention (only if enabled)
        if self.use_attention:
            decoder_outputs = self.decoder_attention(encoder_outputs, decoder_outputs)



        """
        End Task 1 2/2
        """

        # 5) Projection
        decoder_outputs = self.decoder_dense(decoder_outputs)  # [batch, max_tgt_len, vocab_target_size]
        return decoder_outputs

    def decode_step(self, target_words, decoder_states, encoder_outputs):
        """
        A single step of decoder inference:
          - Embedding for the current token
          - One-step LSTM forward
          - (Optional) attention over encoder outputs
          - Project to vocab
        Inputs:
          tgt_input: shape [batch_size, 1]
          decoder_states: (dec_h, dec_c) each is [1, batch_size, hidden_size]
          encoder_outputs: [batch_size, max_src_len, hidden_size]
        Returns:
          logits for the next token, and the new decoder states
        """


           # ---------------------------------------------------------
        # Task 2: Decoder inference (ONE step)
        # Follow template steps:
        #  1) Embedding for the current token
        #  2) One-step LSTM forward (using previous decoder_states)
        #  3) (Optional) attention over encoder outputs
        #  4) Project to vocab (logits)
        # ---------------------------------------------------------

        # decoder_states is a tuple: (dec_h, dec_c)
        # each has shape [1, batch_size, hidden_size]
        dec_h, dec_c = decoder_states

        # 1) Embedding for the current token
        # target_words has shape [batch_size, 1]
        # after embedding -> [batch_size, 1, embedding_size]
        target_words_embeddings = F.dropout(
            self.embedding_target(target_words),
            p=self.embedding_dropout_rate,
            training=self.training
        )

        # 2) One-step LSTM forward
        # We feed ONE token at a time, so the output length is 1.
        # decoder_out -> [batch_size, 1, hidden_size]
        decoder_out, (dec_h, dec_c) = self.decoder_lstm(
            target_words_embeddings,
            (dec_h, dec_c)
        )

        # 3) Optional attention over encoder outputs
        # If attention is ON, this returns [batch_size, 1, hidden_size*2]
        if self.use_attention:
            decoder_out = self.decoder_attention(encoder_outputs, decoder_out)

        # 4) Project to vocab (logits)
        # logits -> [batch_size, 1, vocab_target_size]
        decoder_outputs = self.decoder_dense(decoder_out)


        return decoder_outputs, (dec_h, dec_c)

    def encode(self, source_words):
        """
        Encode the source sequence once for inference.
        """
        source_words_embeddings = F.dropout(self.embedding_source(source_words), p=self.embedding_dropout_rate, training=False)
        encoder_outputs, (enc_h, enc_c) = self.encoder_lstm(source_words_embeddings)
        return encoder_outputs, (enc_h, enc_c)

## NmtModel, `time_used()` method: outputs the time differences between the current time and the input time.
It is always good practice to record the time usage of an individual process, so you always know which part is most expensive to run.

In [11]:
class NmtModel(NmtModel):
  def time_used(self, start_time):
          """
          Outputs the time differences between now and start_time.
          """
          curr_time = time.time()
          used_time = curr_time - start_time
          m = int(used_time // 60)
          s = int(used_time - 60 * m)
          return f"{m} m {s} s"

## The `get_target_sentences()` method takes sentence indices and returns the string tokens.
The method is a helper for the `eval_process` method, which is used to create reference and candidate sentences for evaluation.



In [12]:
class NmtModel(NmtModel):
    def get_target_sentences(self, sents, vocab):
        """
        Convert a batch of sequences of token-IDs into strings.
        Stop at <end> or skip <start>.
        """
        str_sents = []
        num_sent, max_len = sents.shape
        for i in range(num_sent):
            str_sent = []
            for j in range(max_len):
                t = int(sents[i, j])
                if t == self.SOS:
                    continue
                if t == self.EOS:
                    break
                str_sent.append(vocab[t])
            str_sents.append(" ".join(str_sent))
        return str_sents

## NmtModel, `eval_process()` method: runs evaluation on the given dataset.
The method first translates the source sentences into the target language, and then compares them to the reference sentences. As a result, it outputs standard BLEU scores (as computed by the state-of-the-art Sacrebleu (https://github.com/mjpost/sacrebleu) implementation). Note that here we do not tokenise our outputs and references as they are already tokenised and we compare the models internally. However, to ensure comparability to other published work for the same data you need to detokenise your outputs and then use the default tokenisation with the argument `tokenize=BLEU.TOKENIZER_DEFAULT`.

*`eval()` method and exist in `nn.module`, used to convert the model into a evaluation mode. for example turning off dropout*

In [13]:
class NmtModel(NmtModel):
    def eval_process(self, dataset):
        """
        Evaluate on a given dataset, returning a BLEU score.
        """
        self.eval()  # set model to eval mode, turning off dropout

        source_words, target_words_labels = dataset
        device = next(self.parameters()).device

        # Convert to torch
        source_words_torch = torch.LongTensor(source_words).to(device)
        target_words_labels_torch = torch.LongTensor(target_words_labels).to(device)

        # 1) Encode
        with torch.no_grad():
            encoder_outputs, (enc_h, enc_c) = self.encode(source_words_torch)

        batch_size = source_words_torch.size(0)
        # Start tokens => shape [batch_size, 1]
        step_tgt = torch.LongTensor([self.SOS]*batch_size).unsqueeze(1).to(device)
        decoder_states = (enc_h, enc_c)

        predictions = []
        # 2) decode up to max_target_step
        for _ in range(self.max_target_step):
            with torch.no_grad():
                logits, decoder_states = self.decode_step(step_tgt, decoder_states, encoder_outputs)
            # argmax over vocab
            step_tgt = torch.argmax(logits, dim=-1)  # [batch_size, 1]
            predictions.append(step_tgt.cpu().numpy())

        # Convert predictions => [batch, max_target_step]
        predictions = np.concatenate(predictions, axis=1)
        # Convert to strings
        candidates = self.get_target_sentences(predictions, self.target_dict.vocab)
        references = self.get_target_sentences(target_words_labels_torch.cpu().numpy(), self.target_dict.vocab)

        # Score with sacrebleu
        score = corpus_bleu(candidates, [references], tokenize='none').score
        print(f"Model BLEU score: {score:.2f}")
        return score

## The `train_main` method starts the training.
Please note you will need to change the argument of the `use_attention` parameter accordingly.


In [14]:
class NmtModel(NmtModel):
    def train_model(self, train_data, dev_data, test_data, epochs=10, lr=0.01, clip_norm=5.0, device='cpu'):
        """
        Oversees the training process.
        1) For each epoch, train on the entire training dataset.
        2) Evaluate on dev data after each epoch.
        3) Finally evaluate on test data.
        """
        self.to(device)
        optimizer = optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss(ignore_index=self.target_dict.PAD)

        # Unpack data
        source_words_train, target_words_train, target_words_train_labels = train_data
        source_words_dev,   target_words_dev_labels = dev_data
        source_words_test,  target_words_test_labels = test_data

        # For convenience, convert all to torch on CPU first
        source_words_train_torch = torch.LongTensor(source_words_train)
        target_words_train_torch = torch.LongTensor(target_words_train)
        target_words_train_labels_torch = torch.LongTensor(target_words_train_labels.squeeze(-1))  # [batch, max_len]

        # We won't build a fancy DataLoader here; just run with entire batch or smaller mini-batches
        num_samples = source_words_train_torch.size(0)
        idx_list = np.arange(num_samples)

        start_time = time.time()

        for epoch in range(1, epochs+1):
            print(f"Starting training epoch {epoch}/{epochs}")
            epoch_time = time.time()

            # Shuffle data
            np.random.shuffle(idx_list)

            # Mini-batch training
            self.train()  # set model to train mode
            batch_size = self.batch_size
            for start_idx in range(0, num_samples, batch_size):
                end_idx = start_idx + batch_size
                excerpt = idx_list[start_idx:end_idx]

                src_batch = source_words_train_torch[excerpt].to(device)
                tgt_batch = target_words_train_torch[excerpt].to(device)
                tgt_labels_batch = target_words_train_labels_torch[excerpt].to(device)

                optimizer.zero_grad()

                logits = self.forward(src_batch, tgt_batch)  # [batch, tgt_len, vocab_size]

                # Flatten for cross entropy
                # logits: [batch*tgt_len, vocab_size]
                # labels: [batch*tgt_len]
                logits_2d = logits.view(-1, logits.size(-1))
                labels_2d = tgt_labels_batch.view(-1)

                loss = loss_fn(logits_2d, labels_2d)
                loss.backward()

                # Clip gradients
                torch.nn.utils.clip_grad_norm_(self.parameters(), clip_norm)
                optimizer.step()

            print(f"Time used for epoch {epoch}: {self.time_used(epoch_time)}")

            # Evaluate on dev
            print(f"Evaluating on dev set after epoch {epoch}/{epochs}:")
            self.eval_process([source_words_dev, target_words_dev_labels])

        # Training finished
        print("Training finished!")
        print(f"Time used for training: {self.time_used(start_time)}")

        # Evaluate on test set
        print("Evaluating on test set:")
        self.eval_process([source_words_test, target_words_test_labels])

In [15]:
def main(source_path, target_path, use_attention=True):
    max_example = 30000
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("loading dictionaries...")
    train_data, dev_data, test_data, source_dict, target_dict = load_dataset(
        source_path, target_path, max_num_examples=max_example
    )
    print(f"read {len(train_data[0])}/{len(dev_data[0])}/{len(test_data[0])} train/dev/test batches")

    # Create model
    model = NmtModel(source_dict, target_dict, use_attention=use_attention)
    # Train
    
    model.train_model(train_data, dev_data, test_data, epochs=10, lr=0.01, clip_norm=5.0, device=device)
    return model, test_data


In [16]:
def ids_to_source_sentence(ids, source_vocab, pad_id=0):
    """
    Convert a sequence of Vietnamese token IDs into a readable sentence.
    Stops at PAD token.
    """
    words = []
    for t in ids:
        if int(t) == pad_id:
            break
        words.append(source_vocab[int(t)])
    return " ".join(words)


def show_translation_samples_with_source(model, source_words_np, target_labels_np, num_samples=5):
    """
    Display sample translations:
      - Source (Vietnamese)
      - Reference (English ground truth)
      - Prediction (Model output)
    """

    model.eval()
    device = next(model.parameters()).device

    # Take first num_samples sentences
    src_np = source_words_np[:num_samples]
    src = torch.LongTensor(src_np).to(device)

    ref = target_labels_np[:num_samples]

    # 🔹 Shape safety: if ref is [B, T, 1], squeeze to [B, T]
    if len(ref.shape) == 3:
        ref = ref.squeeze(-1)

    # Encode source sentences
    with torch.no_grad():
        encoder_outputs, (enc_h, enc_c) = model.encode(src)

    batch_size = src.size(0)

    # Start decoding with <start> token
    step_tgt = torch.LongTensor([model.SOS] * batch_size).unsqueeze(1).to(device)
    decoder_states = (enc_h, enc_c)

    preds = []

    # Generate tokens step-by-step
    for _ in range(model.max_target_step):
        with torch.no_grad():
            logits, decoder_states = model.decode_step(
                step_tgt, decoder_states, encoder_outputs
            )

        step_tgt = torch.argmax(logits, dim=-1)  # [batch_size, 1]
        preds.append(step_tgt.cpu().numpy())

        # 🔹 Early stop if all sentences predicted <end>
        if torch.all(step_tgt.squeeze(1) == model.EOS):
            break

    preds = np.concatenate(preds, axis=1)

    # Convert IDs to readable English sentences
    pred_str = model.get_target_sentences(preds, model.target_dict.vocab)
    ref_str  = model.get_target_sentences(ref,  model.target_dict.vocab)

    print("\n===== SAMPLE TRANSLATIONS =====")
    for i in range(num_samples):
        vi_sent = ids_to_source_sentence(
            src_np[i],
            model.source_dict.vocab,
            pad_id=model.source_dict.PAD
        )

        print(f"\nExample {i+1}")
        print("Source (VI):    ", vi_sent)
        print("Reference (EN): ", ref_str[i])
        print("Prediction (EN):", pred_str[i])


## Task 1: Implement the Embedding Layers and the Encoder
In this task, you will work at the beginning of the `__init__()` and `forward()` method. You will need to first create two `nn.Embedding` layers (one for the source language and one for the target language). Then pass the source embedding into an `nn.LSTM` layer.

Let’s first look at the inputs. You have in total two inputs:
- `source_words`: the word indices of the sentences in the source language. This input has the shape `[batch_size, max_source_sent_len]` during both training and inference.
- `target_words`: the word indices of the sentences in the target language. During training, this input will have the shape `[batch_size, max_target_sent_len]`, but during the inference, it will have the shape `[batch_size, 1]`.

You will need to first create two `nn.Embedding` layers `embedding_source` and `embedding_target`. The Embedding layers will randomly initialise the embeddings for individual words in the vocabulary and the embeddings will be trained together with the network.  The `nn.Embedding` layers have an `input_dim` of the `vocab_size` and an `output_dim` of the `embedding_size`.  Please note the `vocab_size` for the source and the target language are different. Also, you will need to set the `padding_idx` in order to ignore the paddings.
  
Secondly, you need to look up the embeddings for the current inputs (`source_words` and `target_words`) by passing them through the `nn.Embedding` layers you created. The embeddings for source and target words need to be called `source_words_embeddings` and `target_words_embeddings` respectively.

Thirdly, you can create an `nn.LSTM` layer to process the `source_words_embeddings`, you will need to set the `bidirectional` to `False` and set the `batch_first` to `True`.


## Task 2: Implement the Decoder for inference
In this task, you will work on the `decode_step()` method.

The decoder for inference is similar to the encoder for training but it only performs one step of the decoding at a time. Remember the decoders share all the layers, you will need to use the layers created in the decoder for training. In total three layers are used in both decoders. These are the `decoder_lstm` (the decoder nn.LSTM layer), `decoder_dense` (the decoder final layer) and the `decoder_attention` (the attention layer for the attention based model) layers.

First, unlike the decoder for training that uses the `encoder_states` (`enc_h`, `enc_c`) as the `hidden_size` for `decoder_lstm`, we need to use the decoder states from the previous step instead (`dec_h`, `dec_c`).  You need to put them together in a list to create the `decoder_states`. If you take a look at the `eval_process` method you will find out that for the first step, the `decoder_states` passed into the model are actually the `encoder_states` (same as in the decoder during training), while in the subsequent steps the `decoder_states` become the ones the `decoder_states` returns in the previous step.

Secondly, you will need to pass the `target_word_embeddings` and `decoder_states` to the `decoder_lstm`.

Thirdly you will write an if statement for the attention model just like we did in the decoder for training.

Finally, pass the output of the nn.LSTM (for basic model) or the attention layer (for attention model) into the final linear layer of the decoder (`decoder_dense`) to get probabilities for the next token.

You have now a functional NMT system, why not test it out to see how well it works. Please note you need to set the `use_attention` to `False` since you haven’t implemented the attention layer yet.  The system will take about a minute to finish 10 epochs of training and you will get a BLEU score of around 4.


In [17]:
# Train basic model
model, test_data = main(SOURCE_PATH, TARGET_PATH, use_attention=False)

source_words_test, target_words_test_labels = test_data
show_translation_samples_with_source(
    model,
    source_words_test,
    target_words_test_labels,
    num_samples=5
)


loading dictionaries...


[HAMI-core Msg(958:140078627843392:libvgpu.c:855)]: Initialized


read 24000/3000/3000 train/dev/test batches
number of tokens in source: 2034, number of tokens in target: 2506


/opt/conda/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Starting training epoch 1/10
Time used for epoch 1: 0 m 11 s
Evaluating on dev set after epoch 1/10:


...

Time used for epoch 8: 0 m 12 s
Evaluating on dev set after epoch 8/10:
Model BLEU score: 3.49
Starting training epoch 9/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 9: 0 m 14 s
Evaluating on dev set after epoch 9/10:
Model BLEU score: 3.96
Starting training epoch 10/10
Time used for epoch 10: 0 m 20 s
Evaluating on dev set after epoch 10/10:


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Model BLEU score: 4.18
Training finished!
Time used for training: 2 m 16 s
Evaluating on test set:
Model BLEU score: 4.69

===== SAMPLE TRANSLATIONS =====

Example 1
Source (VI):     trích dẫn thứ hai đến từ người đứng đầu cơ quan quản lý dịch vụ tài chính vương quốc anh .
Reference (EN):  the second quote is from the head of the u.k. financial services <unk> .
Prediction (EN): the <unk> is <unk> .

Example 2
Source (VI):     chuyện trở nên tồi tệ hơn .
Reference (EN):  it gets worse .
Prediction (EN): and the <unk> was <unk> .

Example 3
Source (VI):     chuyện gì đang diễn ra ở đây ? sao chuyện này lại có thể ?
Reference (EN):  what &apos;s happening here ? how can this be possible ?
Prediction (EN): what &apos;s the <unk> <unk> ?

Example 4
Source (VI):     thật không may , câu trả lời là đúng vậy đấy .
Reference (EN):  unfortunately , the answer is yes .
Prediction (EN): it &apos;s a <unk> <unk> .

Example 5
Source (VI):     nhưng mà , có một giải pháp rất thú vị đến từ lĩnh vực đư

## Task 3: Implement the Attention layer
In this task, you will work on the `forward` method of the `AttentionLayer` class.

The attention decoder is the secret recipe for the success of the NMT. It enables the decoder to access all the encoder outputs and focus on their different parts during different steps. By contrast, the basic model only has access to the final states of the encoder. There are a few different ways to build an attention mechanism. Here we build an attention mechanism similar to the one proposed by Luong et al. (2015), which computes the score between `decoder_outputs` and `encoder_outputs` by dot product.

First, let’s take a look at the shape of our inputs (`encoder_outputs, decoder_outputs`). `encoder_outputs` has a shape of `[batch_size, max_source_sent_len, hidden_size]`. `decoder_outputs` has a shape of `[batch_size, max_target_sent_len, hidden_size]`. In order to multiply them, we need to first transpose the last two dimensions of `decoder_outputs` to make its shape become `[batch_size, hidden_size, max_target_sent_len]`. You will need to use the backend `permute_dimensions` method to do this.

Once the `decoder_output` is transposed we use the `batch_dot` to compute the dot product. Let’s call the output `luong_score`. It has a shape of `[batch_size, max_source_sent_len, max_target_sent_len]` then you need apply a softmax to the dimension that have a size of `max_source_sent_len` to create an attention score for the `encoder_outputs`.   

Finally, we are going to create the `encoder_vector` by doing element-wise multiplication between the `encoder_outputs` and their attention scores (`luong_score`). But as you may have noticed the shape of `luong_score` is actually not the same as that of `encoder_outputs`, so we need to use the `expand_dims` method to expand dimensions for both of them. For  `luong_score`, you need to expand the last dimension to accommodate the `hidden_size` dimension of `encoder_outputs`. So after expansion, the shape becomes `[batch_size, max_source_sent_len, max_target_sent_len, 1]`. For  `encoder_outputs`, the target shape is `[batch_size, max_source_sent_len, 1, hidden_size]`. When multiplying between the two tensors, the expanded dimensions will be broadcasted so that they have the same shape. The last step is to sum along the `max_source_sent_len` dimension to create the `encoder_vector`.

Before returning the `new_decoder_outputs` we concatenate the `decoder_outputs` and the `encoder_vector` using the concatenate method (the code is already provided).

You’ve created an attention NMT system, let’s run your code (remember to set `use_attention` to True), it will take about a minute on a GPU to train it and you will get a much better BLEU score, usually above 12 (three times better than the score for the basic version).

In [ ]:
model_attn, test_data = main(SOURCE_PATH, TARGET_PATH, use_attention=True)

source_words_test, target_words_test_labels = test_data
show_translation_samples_with_source(
    model_attn,
    source_words_test,
    target_words_test_labels,
    num_samples=5
)

loading dictionaries...
read 24000/3000/3000 train/dev/test batches
number of tokens in source: 2034, number of tokens in target: 2506
Starting training epoch 1/10


/opt/conda/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 1: 0 m 17 s
Evaluating on dev set after epoch 1/10:
Model BLEU score: 9.65
Starting training epoch 2/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 2: 0 m 18 s
Evaluating on dev set after epoch 2/10:
Model BLEU score: 12.35
Starting training epoch 3/10


...

Time used for epoch 4: 0 m 20 s
Evaluating on dev set after epoch 4/10:
Model BLEU score: 13.88
Starting training epoch 5/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 5: 0 m 19 s
Evaluating on dev set after epoch 5/10:
Model BLEU score: 14.75
Starting training epoch 6/10


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Time used for epoch 6: 0 m 17 s
Evaluating on dev set after epoch 6/10:
Model BLEU score: 14.68
Starting training epoch 7/10
